# Idealista

* web_id: idealista id
* url: idealista link
* title: english title
* type: property type
* price: price to rent the property
* deposit: deposit needed to rent
* private_owner: if the property owner is private or not
* professional_name: agency name, null => private owner
* floor_built: built area in square meters. Superficie total
* floor_area: living area in square meters. Superficie util
* floor: floor number
* year_built: year of the building
* orientation: orientation of the property
* bedrooms: number of bedrooms
* bathrooms: number of bathrooms
* second_hand: if the property is not new or not
* lift: if the property has lift or not
* garage_included: if the property has garage or not
* furnished: if the property is furnished or not
* equipped_kitchen: if the property has equipped kitchen or not
* fitted_wardrobes: if the property has fitted wardrobes or not
* air_conditioning: if the property has air conditioning or not
* terrace: if the property has terrace or not
* balcony: if the property has balcony or not
* storeroom: if the property has storeroom or not
* swimming_pool: if the property has swimming pool or not
* garden_area: if the property has a garden area or not
* location: property address
* district: property district from idealista
* subdistrict: property subdistrict from idealista
* postalcode: property postal code
* last update: last update date from idealista

Para arreglar columnas relacionadas con direcciones se puede recurrir a APIs de geolocalización:

* Nominatim:
    * https://nominatim.openstreetmap.org/ui/search.html
    * https://nominatim.org/
* Google maps API:
    * https://developers.google.com/maps/documentation/geocoding/overview?hl=es-419

In [338]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [339]:
df = pd.read_csv('idealista.csv', encoding='utf-8').sample(1000, random_state=42)
df = df.drop(['web_id', 'url', 'title', 'last_update'], axis=1)
df.dropna(subset=['price'])
df.head(2)

,type,price,deposit,private_owner,professional_name,floor_built,floor_area,floor,year_built,orientation,...,air_conditioning,terrace,balcony,storeroom,swimming_pool,garden_area,location,district,subdistrict,postalcode
1183,Flat,2300,NaN,True,NaN,135,NaN,2nd,NaN,north,...,True,False,False,False,False,False,"Avenida de Menéndez Pelayo, 79, Subdistrict Ni...",Retiro,Niño Jesús,28007.0
1038,Flat,4200,NaN,False,TOP TEAM REAL ESTATE,268,NaN,4th,NaN,NaN,...,True,True,True,True,True,False,", Subdistrict Mirasierra, District Fuencarral,...",Fuencarral,Mirasierra,NaN


In [340]:
# idea, crear columna con el número de viviendas a la venta/alquiler de la empresa que alquila la casa
# esta idea está implementada en el notebook 01.calculadora_precio_viviendas.ipynb

In [341]:
# df = df.dropna()
# df.shape # (410, 28)

In [342]:
# floor_built superficie total
# floor_Area superficie util
# supongamos que util es el 85 % de la total
# Opción 1: imputarla si sabemos que se puede derivar de otra columna:
df['floor_area'] = df['floor_area'].fillna(df['floor_built'] * 0.85)


# Opción 2: directamente borrar la columna ya que es practicamente igual que superficie total
# df = df.drop(['floor_area'], axis=1)

# Opción 3: imputarla con imputer sofisticado por ejemplo IterativeImputer, como no hay tantos nulos puede tener sentido imputarla así con esta estrategia

In [343]:
df_subdistricts = df['location'][df['subdistrict'].isna()]
df_subdistricts.iloc[0]

', District Zona Monte el Pilar, Majadahonda, Zona noroeste, Madrid'

In [344]:
df.isna().sum()

type                   0
price                  0
deposit              425
private_owner          0
professional_name    177
floor_built            0
floor_area             0
floor                 34
year_built           698
orientation          527
bedrooms               0
bathrooms              0
second_hand            0
lift                   0
garage_included        0
furnished              0
equipped_kitchen       0
fitted_wardrobes       0
air_conditioning       0
terrace                0
balcony                0
storeroom              0
swimming_pool          0
garden_area            0
location               0
district              13
subdistrict           78
postalcode           256
dtype: int64

In [345]:
df['floor'].unique()

array(['2nd', '4th', '3rd', '1st', nan, '6th', '9th', '5th', '4', '10th',
       'ground', '7th', 'floor', '8th', '11th', '295', '344', '2', '3',
       '160', '93', '18th', '380', '230', '13th', '62', '55', '197', '50',
       '1,000', '280', '309', '97', '191', '20th', '12th', '180', '83',
       '105', '670', '300', '246', '77', '200', '1', '70', '308', '16th',
       '121', '220', '58', '240', '17th', '427', '285'], dtype=object)

In [346]:
df.shape

(1000, 28)

In [347]:
# considerar que si floor es null entonces es porque es una casa o un chalet y no tiene pisos
df['floor'] = df['floor'].fillna(0)

In [348]:
import pandas as pd
import numpy as np




def clean_floor(floor):
    if pd.isna(floor):  # Manejar NaN
        return np.nan
    
    if isinstance(floor, str):
        floor = floor.lower().strip()  # Convertir a minúsculas y limpiar espacios
        
        # Mapeo de ordinales
        ordinal_map = {'1st': 1, '2nd': 2, '3rd': 3, '4th': 4, '5th': 5, 
                       '6th': 6, '7th': 7, '8th': 8, '9th': 9, '10th': 10,
                       '11th': 11, '12th': 12, '13th': 13, '14th': 14, 
                       '15th': 15, '16th': 16, '17th': 17, '18th': 18, 
                       '19th': 19, '20th': 20, '21st': 21, '22nd': 22, 
                       '23rd': 23, '26th': 26, '27th': 27, '30th': 30, '60th': 60}
        
        if floor in ordinal_map:
            return ordinal_map[floor]
        
        if floor == 'ground':  # Planta baja
            return 0
        
        if floor == 'floor':  # Valor irrelevante
            return np.nan
        
        # Reemplazar comas en números grandes y convertirlos a enteros
        floor = floor.replace(',', '')
        
        if floor.isnumeric():  # Si es un número en string, convertirlo
            #SI aparece un piso con una planta superior a 40 lo pondremos a nan para que se impute
            return int(floor) if int(floor) <= 40 else np.nan
    
    if isinstance(floor, (int, float)):  # Si ya es un número, mantenerlo
        return int(floor)
    
    return np.nan  # Si no se puede interpretar, devolver NaN

print('antes de procesar floor', df.shape)
# Aplicar la función de limpieza
df['floor_cleaned'] = df['floor'].apply(clean_floor)

# Filtrar valores irreales (más de 50 pisos en Madrid no tiene sentido)
# df = df[(df['floor_cleaned'].notna()) & (df['floor_cleaned'] >= 0) & (df['floor_cleaned'] <= 50)]

# df['floor_cleaned'] = df['floor_cleaned'].astype(np.int8)
print('después de procesar floor', df.shape)

antes de procesar floor (1000, 28)
después de procesar floor (1000, 29)


In [349]:
import pandas as pd
import numpy as np
import re

def is_missing(x):
    """ Devuelve True si x es NaN o string vacío/espacios. """
    if pd.isna(x):
        return True
    if isinstance(x, str) and not x.strip():
        return True
    return False

def extract_keyword_value(location_str, keyword):
    """
    Busca en location_str algo como "keyword XXX" (ignorando mayúsculas).
    Retorna 'XXX' si lo encuentra, si no None.
    Ej: keyword='District', si ve "District Casco Antiguo," => "Casco Antiguo".
    """
    if not isinstance(location_str, str):
        return None
    pattern = rf"(?i){keyword}\s+([^,]+)"  # ignora mayúsc/minúsc
    match = re.search(pattern, location_str)
    if match:
        return match.group(1).strip()
    return None

def fallback_from_location(location_str):
    """
    Fallback heurístico: split por comas, ignora trozos numéricos y muy cortos,
    y devuelve uno de los últimos trozos como 'mejor aproximación'.
    """
    if not isinstance(location_str, str):
        return "Unknown_from_location"
    
    parts = [p.strip() for p in location_str.split(",") if p.strip()]
    # Filtrar partes que sean solo dígitos (números) o muy cortas (como '1', '2' etc.)
    filtered = [p for p in parts if not p.isdigit()]

    # Si no quedan trozos, devolvemos Unknown
    if not filtered:
        return "Unknown_from_location"
    
    # Como heurística, elegimos el penúltimo o antepenúltimo. Por ejemplo:
    # "Calle Henares, 1, Camarma de Esteruelas, Corredor del Henares, Madrid"
    # filtered = ["Calle Henares", "Camarma de Esteruelas", "Corredor del Henares", "Madrid"]
    # Quiza el penúltimo sea "Corredor del Henares" (una zona más grande),
    # y el antepenúltimo "Camarma de Esteruelas" (municipio).
    # Depende de lo que quieras llamar "district" vs "subdistrict".
    #
    # Ej: supondremos que es mejor tomar el segundo o tercer trozo desde el final:
    if len(filtered) >= 3:
        return filtered[-3]  # antepenúltimo trozo
    else:
        # Si no hay tantos trozos, devolvemos el primero (o último)
        return filtered[0]

def fill_district_subdistrict(row):
    """
    Rellena district y subdistrict si están vacíos, usando:
    1) la extracción por keyword
    2) fallback si no hay keyword
    """
    loc = row['location']

    # DISTRICT
    if is_missing(row['district']):
        # 1) Buscar con la palabra 'District'
        dist_candidate = extract_keyword_value(loc, "District")
        if dist_candidate:
            row['district'] = dist_candidate
        else:
            # 2) Fallback
            row['district'] = fallback_from_location(loc)

    # SUBDISTRICT
    if is_missing(row['subdistrict']):
        # 1) Buscar con la palabra 'Subdistrict'
        subdist_candidate = extract_keyword_value(loc, "Subdistrict")
        if subdist_candidate:
            row['subdistrict'] = subdist_candidate
        else:
            # 2) Fallback
            row['subdistrict'] = fallback_from_location(loc)

    return row

print('antes', df['district'].isna().sum())
print('antes', df['subdistrict'].isna().sum())
# Aplicamos la función fila a fila (puede hacerse con df.apply)
df[['location', 'district', 'subdistrict']] = df[['location', 'district', 'subdistrict']].apply(fill_district_subdistrict, axis=1)

# Ahora df['district'] y df['subdistrict'] tendrán valores
# extraídos de 'location' donde antes estaban vacíos.
print('despues', df['subdistrict'].isna().sum())
print('despues', df['district'].isna().sum())

antes 13
antes 78
despues 0
despues 0


In [350]:
# algo más sofisticado es usar un servicio de geolocalización y sacar el código postal por location o title o cualquier otra columna
print('antes', df['postalcode'].isna().sum())

df['postalcode'] = df.groupby('subdistrict')['postalcode'] \
    .transform(lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else np.nan)
    
print('despues', df['postalcode'].isna().sum())
df = df.dropna(subset=['postalcode'])

antes 256
despues 16


In [351]:
df.to_csv('idealista_clean.csv')

## Pipeline de regresión

In [352]:
df.head(1)

,type,price,deposit,private_owner,professional_name,floor_built,floor_area,floor,year_built,orientation,...,terrace,balcony,storeroom,swimming_pool,garden_area,location,district,subdistrict,postalcode,floor_cleaned
1183,Flat,2300,NaN,True,NaN,135,114.75,2nd,NaN,north,...,False,False,False,False,False,"Avenida de Menéndez Pelayo, 79, Subdistrict Ni...",Retiro,Niño Jesús,28007.0,2.0


In [353]:
# X = df.drop(['price', 'location'], axis=1)
X = df[['type', 'deposit', 'floor_built', 'floor_area', 'private_owner', 'floor_cleaned', 'year_built', 'terrace',	'balcony', 'storeroom', 'swimming_pool', 'garden_area']]
y = df['price']

In [354]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [355]:
numerical_columns = X_train.select_dtypes(include=[np.number]).columns # np.number alternativa
print('numerical_columns', numerical_columns)

categorical_columns = X_train.select_dtypes(exclude=[np.number]).columns
print('\ncategorical_columns',categorical_columns)

numerical_columns Index(['deposit', 'floor_built', 'floor_area', 'floor_cleaned', 'year_built'], dtype='object')

categorical_columns Index(['type', 'private_owner', 'terrace', 'balcony', 'storeroom',
       'swimming_pool', 'garden_area'],
      dtype='object')


In [356]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# pipeline numéricas
numerical_cols = X_train.select_dtypes(include=[np.number]).columns
pipeline_numerical = make_pipeline(
    # IterativeImputer(RandomForestRegressor(), random_state=42, initial_strategy='median'),
    SimpleImputer(strategy='median'),
)

# pipeline categóricas
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns
pipeline_categorical = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    # OneHotEncoder(sparse_output=False, handle_unknown='ignore')
)
# # opción 2 por intentar un imputer más sofisticado 
# categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns
# pipeline_categorical = make_pipeline(
#     OrdinalEncoder(),
#     IterativeImputer(RandomForestClassifier(), random_state=42, initial_strategy='most_frequent'),
# )

# unir pipelines con ColumnTransformer unir cols num y cat
pipeline_all = ColumnTransformer([
    ('numeric', pipeline_numerical, numerical_cols),
    ('categorical', pipeline_categorical, categorical_cols)
])

# pipeline final con el modelo
pipeline = make_pipeline(
    pipeline_all,
    # PCA(),
    RandomForestRegressor()
)
pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['deposit', 'floor_built', 'floor_area', 'floor_cleaned', 'year_built'], dtype='object')),
                                                 ('categorical',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinalencoder',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  Index(['type', 'private_owner', 'terrace', 'balcony', 'storeroom',
       'swimming_pool', 'garden_area'],
      dtype='object'))])),
                ('randomforestregressor', RandomForestRegressor())])

In [357]:
from sklearn.metrics import r2_score

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
r2_score(y_test, y_pred)

0.5003725509905305

## Pipeline de clasificación

In [ ]:
from sklearn.calibration import LabelEncoder

X = df.drop('balcony', axis=1)
y = df['balcony'] # codificarla con LabelEncoder
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded)

In [ ]:
numerical_cols = X_train.select_dtypes(include=[np.number]).columns
pipeline_numerical = make_pipeline(
    # IterativeImputer(RandomForestRegressor(), random_state=42, initial_strategy='median'),
    SimpleImputer(strategy='median'),
)

# pipeline categóricas
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns
pipeline_categorical = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(sparse_output=False)
)
# # opción 2 por intentar un imputer más sofisticado 
# categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns
# pipeline_categorical = make_pipeline(
#     OrdinalEncoder(),
#     IterativeImputer(RandomForestClassifier(), random_state=42, initial_strategy='most_frequent'),
# )

# unir pipelines con ColumnTransformer unir cols num y cat
pipeline_all = ColumnTransformer([
    ('numeric', pipeline_numerical, numerical_cols),
    ('categorical', pipeline_categorical, categorical_cols)
])

# pipeline final con el modelo
pipeline = make_pipeline(
    pipeline_all,
    # PCA(),
    RandomForestClassifier()
)
pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('iterativeimputer',
                                                                   IterativeImputer(estimator=RandomForestRegressor(),
                                                                                    initial_strategy='median',
                                                                                    random_state=42))]),
                                                  Index(['price', 'deposit', 'floor_built', 'floor_area', 'year_built',
       'bedrooms', 'bathrooms', 'postalcode', 'floor_cleaned'],
      dtype='object')),
                                                 ('...
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  Index(['type', 'private_owner', 'professional_name', 'floor', 'orientation',
       'second_hand', 'lift', 'garage_included', 'furnished',
       'equipped_kitchen', 'fitted_wardrobes', 'air_conditioning', 'terrace',
       'storeroom', 'swimming_pool', 'garden_area', 'location', 'district',
       'subdistrict'],
      dtype='object'))])),
                ('randomforestregressor', RandomForestRegressor())])